In [1]:
import pandas as pd


/Users/katevu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:


files = {
    "PGE":  "PGE_Interconnected_Project_Sites_Jan2020-Sep2026.csv",
    "PGE_Historical": "PGE_Interconnected_Project_Sites_Historical-Jan2020.csv",
    "SCE": "SCE_Interconnected_Project_Sites_Jan2020-Sep2026.csv",     # fill in
    "SCE_Historical": "SCE_Interconnected_Project_Sites_Historical-Jan2020.csv",
    "SDGE": "SDGE_Interconnected_Project_Sites_Historical-Sep2026.csv",    # fill in
}

dfs = {}
for name, f in files.items():
    df = pd.read_csv(f"raw_data/{f}", low_memory=False)
    df.columns = df.columns.str.strip()
    dfs[name] = df
    print(name, df.shape)

# Do the columns match across utilities?
base = set(dfs["PGE"].columns)
for name, df in dfs.items():
    print(name, "missing vs PGE:", base - set(df.columns),
          "| extra vs PGE:", set(df.columns) - base)

PGE (610741, 269)
PGE_Historical (409314, 152)
SCE (486929, 269)
SCE_Historical (301209, 152)
SDGE (386301, 269)
PGE missing vs PGE: set() | extra vs PGE: set()
PGE_Historical missing vs PGE: {'Inverter Quantity 57', 'Inverter Model 35', 'Inverter Model 28', 'Inverter Manufacturer 53', 'Inverter Model 61', 'Inverter Model 57', 'Inverter Manufacturer 43', 'Inverter Model 53', 'Inverter Quantity 60', 'Generator Model 8', 'Inverter Model 43', 'Inverter Quantity 40', 'Inverter Manufacturer 49', 'Inverter Manufacturer 52', 'Inverter Manufacturer 34', 'Inverter Model 55', 'Inverter Model 42', 'Inverter Quantity 55', 'Inverter Model 36', 'Inverter Manufacturer 48', 'Inverter Quantity 39', 'Inverter Model 56', 'Inverter Quantity 34', 'Generator Manufacturer 8', 'Inverter Model 30', 'Inverter Model 27', 'Inverter Quantity 30', 'Inverter Manufacturer 28', 'Inverter Manufacturer 27', 'Inverter Quantity 29', 'Inverter Quantity 46', 'Inverter Model 51', 'Inverter Model 34', 'Inverter Manufacturer 4

In [4]:
keep = ["Application Id", "Application Status", "Utility", "Service County",
        "Technology Type", "Customer Sector", "App Approved Date",
        "NEMPV or nonNEMPV", "Previous Application"]

dg = pd.concat([df[keep].assign(source_file=name) for name, df in dfs.items()],
               ignore_index=True)

print(dg.shape)
print(dg["source_file"].value_counts())
print(dg["Utility"].value_counts(dropna=False))   # should match source_file
print(dg["Application Id"].duplicated().sum(), "duplicate application IDs")


(2194494, 10)
source_file
PGE               610741
SCE               486929
PGE_Historical    409314
SDGE              386301
SCE_Historical    301209
Name: count, dtype: int64
Utility
PGE     1020053
SCE      788136
SDGE     386300
NaN           5
Name: count, dtype: int64
0 duplicate application IDs


In [5]:
# How many rows are actually solar, residential, and new?
dg["is_pv"] = dg["Technology Type"].str.contains("photovoltaic|solar", case=False, na=False)
print(dg.groupby("Utility")["is_pv"].mean())                       # share that is PV
print(dg.groupby("Utility")["Customer Sector"].value_counts(normalize=True).round(2))
print(dg.groupby("Utility")["Previous Application"].value_counts(dropna=False))
print(dg.groupby("Utility")["Application Status"].value_counts(dropna=False))
print(dg["Application Id"].duplicated().sum(), "duplicate application IDs")

# Where does it fall in time?
dg["approved"] = pd.to_datetime(dg["App Approved Date"], errors="coerce")
print(dg.groupby("Utility")["approved"].agg(["min", "max", lambda s: s.isna().mean()]))
print(dg.groupby(["Utility", dg["approved"].dt.year]).size().unstack(0))

Utility
PGE     0.962421
SCE     0.997027
SDGE    0.853399
Name: is_pv, dtype: float64
Utility  Customer Sector 
PGE      Residential         0.97
         Commercial          0.02
         Educational         0.00
         Non-Profit          0.00
         Industrial          0.00
         Military            0.00
         Other Government    0.00
SCE      Residential         0.98
         Commercial          0.01
         Not Available       0.00
         Educational         0.00
         Non-Profit          0.00
         Industrial          0.00
         Other Government    0.00
         Military            0.00
SDGE     Residential         0.98
         Commercial          0.02
         Educational         0.00
         Other Government    0.00
         Military            0.00
         Industrial          0.00
Name: proportion, dtype: float64
Utility  Previous Application
PGE      NaN                     928441
         Yes                      91612
SCE      NaN                  

In [6]:
d = dg[dg["is_pv"]]
d = d[d["Customer Sector"].str.strip().str.lower() == "residential"]
d = d[d["NEMPV or nonNEMPV"].str.contains("^NEM", case=False, na=False)]

# Always print row counts for your Methods section
print(len(dg), "->", len(d))
print(d.groupby("Utility").size())

total_thru_2022 = d[d["approved"].dt.year <= 2022]
new_2020_2022  = d[d["approved"].dt.year.between(2020, 2022)]
total_thru_2026 = d

for name, x in [("thru2022", total_thru_2022), ("new2020_22", new_2020_2022)]:
    print(name, x.groupby("Utility").size().to_dict())

2194494 -> 2046136
Utility
PGE     953106
SCE     770236
SDGE    322794
dtype: int64
thru2022 {'PGE': 646234, 'SCE': 491301, 'SDGE': 238961}
new2020_22 {'PGE': 254042, 'SCE': 198963, 'SDGE': 85065}


In [13]:
import pandas as pd, glob, os

paths = sorted(glob.glob("raw_data/LEAD_data/**/*.csv", recursive=True))
for p in paths:
    try:
        df = pd.read_csv(p, dtype=str, nrows=3)      # no low_memory=False here
        print("OK  ", os.path.basename(p), len(df.columns), "columns")
    except Exception as e:
        print("FAIL", os.path.basename(p), "->", str(e)[:100])

OK   CA AMI Census Tracts 2022.csv 29 columns
OK   CA AMI Counties 2022.csv 31 columns
OK   CA FPL Census Tracts 2022.csv 29 columns
OK   CA FPL Counties 2022.csv 31 columns
OK   CA LLSI Census Tracts 2022.csv 29 columns
OK   CA LLSI Counties 2022.csv 31 columns
OK   CA SMI Census Tracts 2022.csv 29 columns
OK   CA SMI Counties 2022.csv 31 columns


In [14]:

def county_summary(path):
    df = pd.read_csv(path, dtype={"FIP": str}, low_memory=False)
    df["FIP"] = df["FIP"].str.zfill(5)

    num = ["UNITS", "HINCP*UNITS", "ELEP*UNITS", "GASP*UNITS", "FULP*UNITS"]
    df[num] = df[num].apply(pd.to_numeric, errors="coerce")
    print(path, "| missing values:\n", df[num].isna().sum().to_dict())

    df["spend"] = df[["ELEP*UNITS", "GASP*UNITS", "FULP*UNITS"]].sum(axis=1)
    g = (df.groupby(["FIP", "NAME"])
           .agg(households=("UNITS", "sum"),
                income=("HINCP*UNITS", "sum"),
                spend=("spend", "sum"))
           .reset_index())
    g["energy_burden_pct"] = 100 * g["spend"] / g["income"]
    g["county"] = g["NAME"].str.replace(r"\s+County$", "", regex=True)
    return g

ami = county_summary("raw_data/LEAD_data/CA AMI Counties 2022.csv")
print(ami.shape)                     # expect 58 rows
print(ami["households"].sum())       # expect roughly 13 million
print(ami["energy_burden_pct"].describe())
print(ami.sort_values("energy_burden_pct")[["county", "households", "energy_burden_pct"]].to_string())

raw_data/LEAD_data/CA AMI Counties 2022.csv | missing values:
 {'UNITS': 0, 'HINCP*UNITS': 0, 'ELEP*UNITS': 0, 'GASP*UNITS': 0, 'FULP*UNITS': 0}
(58, 7)
13749689.838532314
count    58.000000
mean      2.492872
std       0.862914
min       0.726683
25%       1.735760
50%       2.719195
75%       3.133522
max       4.215049
Name: energy_burden_pct, dtype: float64
             county    households  energy_burden_pct
37    San Francisco  4.612845e+05           0.726683
42      Santa Clara  7.847633e+05           0.999485
40        San Mateo  3.846406e+05           1.030400
20            Marin  1.419515e+05           1.097085
0           Alameda  5.884632e+05           1.209639
41    Santa Barbara  1.505183e+05           1.328088
29           Orange  1.080594e+06           1.379724
43       Santa Cruz  1.039569e+05           1.452233
36        San Diego  1.154404e+06           1.483072
6      Contra Costa  4.107984e+05           1.552037
39  San Luis Obispo  1.081030e+05           1.580023
